# Fine-Tuning Gemma 3 270M for Function Calling

This notebook demonstrates how to fine-tune the `google/gemma-3-270m-it` model for function calling using LoRA.

In [1]:
#!pip install -q -U transformers accelerate datasets peft trl

In [2]:
import os
from enum import Enum
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

set_seed(42)

Skipping import of cpp extensions due to incompatible torch version 2.13.0+cu130 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [3]:
class ChatmlSpecialTokens(str, Enum):
    tools = "<tools>"
    eotools = "</tools>"
    think = "<think>"
    eothink = "</think>"
    tool_call = "<tool_call>"
    eotool_call = "</tool_call>"
    tool_response = "<tool_response>"
    eotool_response = "</tool_response>"
    pad_token = "<pad>"
    eos_token = "<eos>"

    @classmethod
    def list(cls):
        return [c.value for c in cls]

In [4]:
class Config:
    model_name = "google/gemma-3-270m-it"
    dataset_name = "lmassaron/hermes-function-calling-v1"
    output_dir = "gemma-3-270M-it-function_calling"
    
    lora_arguments = {
        "r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": [
            "embed_tokens", "q_proj", "k_proj", "v_proj",
            "gate_proj", "up_proj", "down_proj", "o_proj", "lm_head"
        ],
        "bias": "none",
    }
    
    training_arguments = {
        "num_train_epochs": 1,
        "max_steps": -1,
        "per_device_train_batch_size": 1,
        "per_device_eval_batch_size": 1,
        "gradient_accumulation_steps": 4,
        "max_length": 2048,
        "packing": False,
        "optim": "adamw_torch_fused",
        "learning_rate": 1e-4,
        "loss_type": "nll", 
        "torch_empty_cache_steps":1,
        # we are telling SFTTrainer to bypass the new chunked memory optimization and fall back to the standard PyTorch Cross-Entropy Loss (nll).
        # This standard loss calculation is fully compatible with our LoRA-wrapped lm_head, allowing the training to run flawlessly
        "weight_decay": 0.1,
        "max_grad_norm": 1.0,
        "lr_scheduler_type": "cosine",
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "eval_strategy": "steps",
        "eval_steps": 100,
        "prediction_loss_only": True,
        "logging_steps": 100,
        "report_to": "none",
    }

config = Config()

In [5]:

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

if device.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8:
    compute_dtype = torch.bfloat16
elif device.type == "mps":
    compute_dtype = torch.bfloat16 
else:
    compute_dtype = torch.float32

print(f"Using device: {device}, dtype: {compute_dtype}")

Using device: cuda, dtype: torch.bfloat16


In [6]:
# Setup Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config.model_name,
    pad_token=ChatmlSpecialTokens.pad_token.value,
    additional_special_tokens=ChatmlSpecialTokens.list(),
)

tokenizer.chat_template = "{{ bos_token }}{% for message in messages %}{% if message['role'] != 'system' %}{{ '<start_of_turn>' + message['role'] + '\n' + message['content'] | trim + '<end_of_turn>\n' }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{'<start_of_turn>model\n'}}{% endif %}"

In [7]:
# Load Model
print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    dtype=compute_dtype,
    low_cpu_mem_usage=True,
    device_map=device, 
)

model.resize_token_embeddings(len(tokenizer))
print("Model loaded and token embeddings resized.")

Loading base model...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded and token embeddings resized.


## Dataset & Training

We will now prepare the dataset and launch the training process using SFTTrainer.

In [8]:
print("Preparing dataset...")
def preprocess_and_filter(sample):
    messages = sample["messages"]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tokens = tokenizer.encode(text, truncation=False)
    
    if len(tokens) <= config.training_arguments["max_length"]:
        return {"text": text, "messages": messages}
    else:
        return None

data = (
    load_dataset(config.dataset_name, split="train")
    .rename_column("conversations", "messages")
    .map(preprocess_and_filter)
    .filter(lambda x: x is not None, keep_in_memory=False)
)

dataset_train = data.train_test_split(test_size=0.1, shuffle=True, seed=0)
train_data = dataset_train["train"]
eval_data = dataset_train["test"]
print(f"Train size: {len(train_data)}, Validation size: {len(eval_data)}")

Preparing dataset...
Train size: 3742, Validation size: 416


## Quantitative Pre-Training Evaluation

We define an exact match evaluation function to test accuracy on a subset of validation data.

In [9]:
from tqdm import tqdm

def evaluate_exact_match(dataset, model, tokenizer, num_samples=100):
    correct_tool, total_tool = 0, 0

    model.eval()
    
    with tqdm(total=num_samples, desc="Evaluating Tool Calling") as pbar:
        for item in dataset:
            conversations = item.get("messages", item.get("conversations"))
            if not conversations or conversations[-1]["role"] != "model":
                continue
            for k, message in enumerate(conversations):
                if k > 0 and "<tool_call>" in message["content"]:
                    break
            else:
                continue

            target_message = conversations[k]["content"].strip()        
            query_messages = conversations[:k]
                    
            # Prepare inputs (single-step tokenize avoids double <bos>)
            inputs = tokenizer.apply_chat_template(
                query_messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            ).to(model.device)

            # Generate output, stopping at <end_of_turn>
            eot_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                eos_token_id=[eot_id, tokenizer.eos_token_id],
                pad_token_id=tokenizer.pad_token_id,
            )
            generated_raw = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=False)

            # Clean outputs for exact match comparison
            generated_clean = generated_raw.replace("<end_of_turn>", "").strip()
            expected_clean = target_message.replace("<end_of_turn>", "").strip()
            
            if expected_clean in generated_clean:
                correct_tool += 1

            total_tool += 1
            pbar.update(1)
            if total_tool == num_samples:
                break
            
    tool_acc = correct_tool / total_tool if total_tool > 0 else 0
    print(f"Tool Calling Exact Match Accuracy: {tool_acc:.2%} ({correct_tool}/{total_tool})")
    print("Example target:", expected_clean)
    print("Example generated:", generated_clean)
    return tool_acc

print("--- QUANTITATIVE PRE-TRAINING EVALUATION ---")
_ = evaluate_exact_match(eval_data, model, tokenizer, num_samples=50)

--- QUANTITATIVE PRE-TRAINING EVALUATION ---


Evaluating Tool Calling: 100%|██████████| 50/50 [00:52<00:00,  1.06s/it]

Tool Calling Exact Match Accuracy: 0.00% (0/50)
Example target: <tool_call>
{'name': 'get_random_number', 'arguments': {'min': 1, 'max': 100}}
</tool_call>
Example generated: ```python
import json

def get_random_number(prompt):
    try:
        response = openai.create(
            model="gpt-3.5-turbo",
            prompt=prompt,
            max_tokens=100,
            steps=5
        )
        return response.data[0]
    except Exception as e:
        print(f"Error calling function: {e}")
        return None
```


In [10]:
model.config.use_cache = False

peft_config = LoraConfig(
    **config.lora_arguments,
    task_type="CAUSAL_LM",
    ensure_weight_tying=True,
)

training_args = SFTConfig(
    output_dir=config.output_dir,
    dataset_text_field="text",
    **config.training_arguments,
    bf16 = compute_dtype == torch.bfloat16,
    fp16 = compute_dtype == torch.float16,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print("Starting training process...")
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


Starting training process...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,0.693352,0.412345,0.388911,0.900127,283180.000000
200,0.413801,0.360286,0.354910,0.909389,569381.000000
300,0.339181,0.344412,0.318921,0.913423,854934.000000
400,0.347135,0.335711,0.332159,0.915229,1141730.000000
500,0.340913,0.326082,0.303412,0.917512,1428281.000000
600,0.362316,0.320855,0.320055,0.918554,1713913.000000
700,0.380184,0.317392,0.315159,0.919362,1999986.000000
800,0.333618,0.315662,0.314156,0.919823,2287769.000000
900,0.331178,0.315065,0.310188,0.919975,2577233.000000
936,0.331178,0.315139,0.310292,0.920020,2677080.000000


/home/lmassaron/code/fine-tuning-workshop/.venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:422: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/home/lmassaron/code/fine-tuning-workshop/.venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:422: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


TrainOutput(global_step=936, training_loss=0.39050883411342263, metrics={'train_runtime': 1664.834, 'train_samples_per_second': 2.248, 'train_steps_per_second': 0.562, 'total_flos': 1807545055357440.0, 'train_loss': 0.39050883411342263, 'epoch': 1.0})

## Quantitative Post-Training Evaluation

Let's evaluate exact match accuracy after fine-tuning.

In [11]:
print("--- QUANTITATIVE POST-TRAINING EVALUATION ---")
_ = evaluate_exact_match(eval_data, model, tokenizer, num_samples=50)

--- QUANTITATIVE POST-TRAINING EVALUATION ---


Evaluating Tool Calling: 100%|██████████| 50/50 [00:32<00:00,  1.56it/s]

Tool Calling Exact Match Accuracy: 94.00% (47/50)
Example target: <tool_call>
{'name': 'get_random_number', 'arguments': {'min': 1, 'max': 100}}
</tool_call>
Example generated: <tool_call>
{'name': 'get_random_number', 'arguments': {'min': 1, 'max': 100}}
</tool_call>
